# Portfolio Optimization Under Uncertainty Using Machine Learning

This notebook is the compact research narrative for the repository. The production-style implementation lives in `src/ml_portfolio/backtest.py`; `run_analysis.py` reproduces the complete walk-forward backtest and figures.

## Research question

Can machine-learning return forecasts improve portfolio allocation when expected returns are uncertain, relative to historical mean-variance optimization, equal weighting, and risk parity?

The key methodological requirement is **strict time ordering**: every monthly portfolio decision may only use information observable at that rebalance date.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
summary = pd.read_csv(ROOT / "results" / "performance_summary.csv", index_col=0)
avg_weights = pd.read_csv(ROOT / "results" / "average_weights.csv", index_col=0)
summary

## Data

Five ETFs represent distinct asset classes: SPY (U.S. equities), EFA (international developed equities), AGG (bonds), GLD (gold), and VNQ (real estate). Daily OHLCV files were obtained from Stooq. The research sample is 2018–2024, with 2023–2024 reserved for walk-forward evaluation.

In [ ]:
from ml_portfolio.backtest import BacktestConfig, load_prices, log_returns_from_prices

config = BacktestConfig()
prices = load_prices(ROOT / "data")
returns = log_returns_from_prices(prices, config.start, config.end)
print(prices.loc[config.start:config.end].shape)
returns.describe().round(4)

## Forecast design

The original coursework predicted daily returns and later averaged predictions over the full test set. For this repository, the target is redesigned to match the monthly holding period. At each date `t`, the Random Forest predicts the cumulative log return over the next 21 trading days using only features known at `t`: five recent daily returns, 21-day momentum, and 21-day realized volatility.

Training observations whose forward 21-day target is not yet fully observed are automatically excluded.

In [ ]:
from ml_portfolio.backtest import make_supervised

example = make_supervised(returns["SPY"], n_lags=5, horizon=21)
example.tail()

## Uncertainty-aware allocation

For each ETF, predictions from individual Random Forest trees form an ensemble. Their standard deviation is used as a **model-uncertainty proxy**. It is not interpreted as a calibrated statistical confidence interval.

The optimizer uses `robust_mu = mean_forecast - λ × tree_dispersion`, then maximizes expected return per unit of covariance risk under long-only, fully invested, 40%-per-asset constraints.

In [ ]:
avg_weights.round(3)

## Walk-forward evaluation

Portfolios are rebalanced monthly. Each rebalance uses a trailing three-year information window, and 10 bps of transaction cost is charged per unit of turnover. Historical MVO, equal-weight, and risk-parity baselines use the same realized test periods; optimized strategies share the same 40% concentration cap.

![Cumulative portfolio performance](../figures/cumulative_performance.svg)

### Out-of-sample results

| Strategy | Total return | Ann. return | Ann. volatility | Sharpe | Max drawdown |
|---|---:|---:|---:|---:|---:|
| **Historical MVO** | **37.47%** | **17.32%** | 9.83% | **1.76** | -8.48% |
| **ML Robust** | 33.85% | 15.76% | 10.32% | 1.53 | -10.01% |
| Equal Weight | 22.70% | 10.82% | 9.46% | 1.14 | -9.28% |
| Risk Parity | 17.84% | 8.59% | **7.90%** | 1.09 | **-8.26%** |

The ML strategy is competitive and improves on the simple diversified baselines, but historical MVO remains strongest in this test window.

In [ ]:
summary.style.format({
    "Total Return": "{:.2%}",
    "Annualized Return": "{:.2%}",
    "Annualized Volatility": "{:.2%}",
    "Sharpe": "{:.2f}",
    "Sortino": "{:.2f}",
    "Max Drawdown": "{:.2%}",
})

## Interpretation

The corrected backtest does **not** support a claim that ML dominates traditional portfolio optimization. Historical MVO produced the strongest total return and risk-adjusted performance over this two-year test, while the uncertainty-aware ML strategy still outperformed equal weighting and risk parity on return and Sharpe.

The more important lesson is methodological: the apparent advantage in the original assignment became smaller once test-period look-ahead and forecast-horizon mismatch were removed. For a financial ML project, that is a useful model-risk result rather than a failure.

## Limitations and next steps

The universe contains only five ETFs and the test window is short. Hyperparameters and the uncertainty penalty are not tuned inside a nested time-series validation loop, the risk-free rate is set to zero for Sharpe reporting, and tree dispersion is only an uncertainty proxy. A stronger follow-up would use nested walk-forward validation, calibrated probabilistic forecasts, a larger asset universe, and a longer out-of-sample period.